# Data Cleaning — Fréquence Hebdomadaire

Variables retenues :
- **WTI** : WTI futures (CL1 Comdty) — Choc exogène énergie
- **HY** : High yield index yield to worst (LF98YW Index) — Indicateur de stress crédit
- **SP500** : S&P 500 (SPX Index) — Marché actions (cible)
- **US2Y** : US 2-year Rate (USGG2YR Index) — Taux sans risque

Pipeline : chargement daily → rééchantillonnage vendredi → log-rendements → excès de rendement → export.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Chargement & Sélection des variables

In [2]:
raw_path = Path("../data/raw/data_hec_projet_1.xlsx")

# Les 5 premières lignes contiennent les métadonnées (noms, tickers, etc.) → on les skippe
# La ligne 5 devient le header, la colonne 0 (Dates) devient l'index
daily_raw = pd.read_excel(raw_path, sheet_name="Daily", skiprows=5, index_col=0, parse_dates=True)

print("Dimensions brutes :", daily_raw.shape)
daily_raw.head(3)

Dimensions brutes : (9443, 13)


,PX_LAST,PX_LAST.1,PX_LAST.2,PX_LAST.3,PX_LAST.4,PX_LAST.5,PX_LAST.6,PX_LAST.7,PX_LAST.8,PX_LAST.9,PX_LAST.10,PX_LAST.11,PX_LAST.12
Dates,,,,,,,,,,,,,
1990-01-01,21.82,19.69,77.8405,NaN,NaN,353.40,567.34,214.69,168.228,7.935,7.841,16.08,401.25
1990-01-02,22.89,19.90,81.0621,NaN,NaN,359.69,568.96,217.30,169.943,7.930,7.875,16.08,399.00
1990-01-03,23.68,20.95,83.0504,NaN,NaN,358.76,569.10,220.45,170.780,7.974,7.927,16.08,395.00


In [3]:
# Sélection des 4 colonnes utiles (positions après skiprows=5 + index_col=0) :
# col 0 = WTI (CL1 Comdty), col 5 = S&P500 (SPX Index),
# col 10 = US 2Y Rate (USGG2YR Index), col 11 = High Yield (LF98YW Index)
df = daily_raw.iloc[:, [0, 5, 10, 11]].copy()
df.columns = ["WTI", "SP500", "US2Y", "HY"]
df.index.name = "Date"

print("Shape après sélection :", df.shape)
df.head()

Shape après sélection : (9443, 4)


,WTI,SP500,US2Y,HY
Date,,,,
1990-01-01,21.82,353.40,7.841,16.08
1990-01-02,22.89,359.69,7.875,16.08
1990-01-03,23.68,358.76,7.927,16.08
1990-01-04,23.41,355.67,7.910,16.08
1990-01-05,23.08,352.20,7.885,16.08


## 2. Rééchantillonnage Hebdomadaire (vendredi)

In [4]:
# W-FRI = week ending Friday ; .last() prend le dernier prix valide de la semaine
# Si vendredi est férié, on remonte au dernier jour de trading disponible cette semaine-là
df_weekly = df.resample("W-FRI").last()

print("Shape hebdomadaire :", df_weekly.shape)
print(f"Période : {df_weekly.index[0].date()} → {df_weekly.index[-1].date()}")
df_weekly.head()

Shape hebdomadaire : (1889, 4)
Période : 1990-01-05 → 2026-03-13


,WTI,SP500,US2Y,HY
Date,,,,
1990-01-05,23.08,352.20,7.885,16.08
1990-01-12,22.13,339.93,7.938,16.08
1990-01-19,22.08,339.15,8.162,16.08
1990-01-26,22.56,325.80,8.263,16.08
1990-02-02,23.02,330.92,8.341,16.53


## 3. Calcul des Rendements & Excès de Rendement

In [ ]:
df_ret = df_weekly.copy()

# Log-rendements continus : np.log(P_t).diff() = log(P_t) - log(P_{t-1})
for col in ["WTI", "SP500", "HY"]:
    df_ret[col] = np.log(df_weekly[col]).diff()

# Taux sans risque hebdomadaire : le US2Y est annualisé (en %) → on divise par 52 
df_ret["US2Y"] = df_weekly["US2Y"] / 52

# Excès de rendement = rendement brut − taux sans risque hebdomadaire
df_ret["SP500_excess"] = df_ret["SP500"] - df_ret["US2Y"]
df_ret["HY_excess"] = df_ret["HY"] - df_ret["US2Y"]

df_ret.head()

,WTI,SP500,US2Y,HY,SP500_excess,HY_excess
Date,,,,,,
1990-01-05,NaN,NaN,0.151635,NaN,NaN,NaN
1990-01-12,-0.042032,-0.035459,0.152654,0.000000,-0.188113,-0.152654
1990-01-19,-0.002262,-0.002297,0.156962,0.000000,-0.159259,-0.156962
1990-01-26,0.021506,-0.040159,0.158904,0.000000,-0.199063,-0.158904
1990-02-02,0.020185,0.015593,0.160404,0.027601,-0.144811,-0.132803


## 4. Nettoyage Final

In [6]:
# Suppression de la première ligne NaN créée par le .diff()
df_clean = df_ret.dropna()

print("Shape finale :", df_clean.shape)
print(f"Période : {df_clean.index[0].date()} → {df_clean.index[-1].date()}")
df_clean.describe().round(4)

Shape finale : (1888, 6)
Période : 1990-01-12 → 2026-03-13


,WTI,SP500,US2Y,HY,SP500_excess,HY_excess
count,1888.0000,1888.0000,1888.0000,1888.0000,1888.0000,1888.0000
mean,0.0007,0.0016,0.0621,-0.0004,-0.0605,-0.0625
std,0.0502,0.0232,0.0434,0.0279,0.0492,0.0509
min,-0.3904,-0.2008,0.0019,-0.1619,-0.2323,-0.2674
25%,-0.0243,-0.0102,0.0182,-0.0112,-0.0962,-0.1006
50%,0.0030,0.0028,0.0647,0.0000,-0.0637,-0.0663
75%,0.0280,0.0142,0.0955,0.0076,-0.0199,-0.0274
max,0.2799,0.1142,0.1720,0.2789,0.1099,0.2728


## 5. Export

In [7]:
# Export du DataFrame propre, aligné et stationnaire
df_clean.to_csv(Path("../data/processed/base_nettoyee_hebdo.csv"))
print("Exporté : ../data/processed/base_nettoyee_hebdo.csv")

Exporté : ../data/processed/base_nettoyee_hebdo.csv


In [8]:
import numpy as np

df_var = df.copy()

# Log-returns pour les variables de prix et de niveau
for col in ["WTI", "SP500"]:
    df_var[col] = np.log(df[col]) - np.log(df[col].shift(1))

# US2Y et HY restent en niveaux (taux / spreads, déjà stationnaires)

# La différence log crée un NaN sur la première ligne
df_var = df_var.dropna()

print("Shape après transformation:", df_var.shape)
print(f"Période : {df_var.index[0].date()} → {df_var.index[-1].date()}")
df_var.head()

Shape après transformation: (9442, 4)
Période : 1990-01-02 → 2026-03-11


,WTI,SP500,US2Y,HY
Date,,,,
1990-01-02,0.047873,0.017642,7.875,16.08
1990-01-03,0.033931,-0.002589,7.927,16.08
1990-01-04,-0.011468,-0.008650,7.910,16.08
1990-01-05,-0.014197,-0.009804,7.885,16.08
1990-01-08,-0.065348,0.004504,7.893,16.08
